In [1]:

import pandas as pd
from pathlib import Path

DATA_DIR = Path.home() / (
    "Library/CloudStorage/"
    "GoogleDrive-zhaozihe17@gmail.com/"
    "我的云端硬盘/momentum_research/data/raw"
)

tickers = [
    "AAPL", "MSFT", "NVDA", "GOOGL", "META", "AMZN",
    "JPM", "BAC", "GS", "V", "MA",
    "JNJ", "UNH", "PFE", "LLY",
    "WMT", "COST", "PG", "KO", "XOM", "SPY"
]

data = {}

for ticker in tickers:
    df = pd.read_csv(
        DATA_DIR / f"{ticker}.csv",
        parse_dates=["Date"],
        index_col="Date"
    )
    data[ticker] = df

print(f"Loaded {len(data)} securities")
print(data["AAPL"].head())


Loaded 21 securities
            Adj Close      Close  Dividends       High        Low       Open  \
Date                                                                           
2014-01-02  17.110128  19.754642        0.0  19.893929  19.715000  19.845715   
2014-01-03  16.734287  19.320715        0.0  19.775000  19.301071  19.745001   
2014-01-06  16.825542  19.426071        0.0  19.528570  19.057142  19.194643   
2014-01-07  16.705208  19.287144        0.0  19.498571  19.211430  19.440001   
2014-01-08  16.810997  19.409286        0.0  19.484285  19.238930  19.243214   

            Stock Splits     Volume  
Date                                 
2014-01-02           0.0  234684800  
2014-01-03           0.0  392467600  
2014-01-06           0.0  412610800  
2014-01-07           0.0  317209200  
2014-01-08           0.0  258529600  


### Data Completeness Check

In [7]:

summary = []

for ticker, df in data.items():
    summary.append({
        "Ticker": ticker,
        "Start Date": df.index.min(),
        "End Date": df.index.max(),
        "Rows": len(df),
        "Missing Values": df.isna().sum().sum(),
        "Duplicate Dates": df.index.duplicated().sum()
    })

summary_df = pd.DataFrame(summary)

display(summary_df)


,Ticker,Start Date,End Date,Rows,Missing Values,Duplicate Dates
0,AAPL,2014-01-02,2025-12-31,3018,0,0
1,MSFT,2014-01-02,2025-12-31,3018,0,0
2,NVDA,2014-01-02,2025-12-31,3018,0,0
3,GOOGL,2014-01-02,2025-12-31,3018,0,0
4,META,2014-01-02,2025-12-31,3018,0,0
5,AMZN,2014-01-02,2025-12-31,3018,0,0
6,JPM,2014-01-02,2025-12-31,3018,0,0
7,BAC,2014-01-02,2025-12-31,3018,0,0
8,GS,2014-01-02,2025-12-31,3018,0,0
9,V,2014-01-02,2025-12-31,3018,0,0


### 交易日历一致性检查

In [ ]:
reference_dates = data["SPY"].index

calendar_results = []

for ticker, df in data.items():
    missing = reference_dates.difference(df.index) # 差集
    extra = df.index.difference(reference_dates)

    calendar_results.append({
        "Ticker": ticker,
        "Missing Dates": len(missing),
        "Extra Dates": len(extra),
        "Calendar Match": df.index.equals(reference_dates)
    })

calendar_df = pd.DataFrame(calendar_results)
display(calendar_df)

,Ticker,Missing Dates,Extra Dates,Calendar Match
0,AAPL,0,0,True
1,MSFT,0,0,True
2,NVDA,0,0,True
3,GOOGL,0,0,True
4,META,0,0,True
5,AMZN,0,0,True
6,JPM,0,0,True
7,BAC,0,0,True
8,GS,0,0,True
9,V,0,0,True


In [ ]:
import exchange_calendars as xcals

nyse = xcals.get_calendar("XNYS")

expected_dates = nyse.sessions_in_range(
    "2014-01-01",
    "2025-12-31"
).tz_localize(None)

actual_dates = data["SPY"].index.normalize()

missing_dates = expected_dates.difference(actual_dates)
extra_dates = actual_dates.difference(expected_dates)

print("Expected sessions:", len(expected_dates))
print("Actual SPY rows:", len(actual_dates))
print("Missing sessions:", missing_dates.tolist())
print("Extra dates:", extra_dates.tolist())

### 与独立的nyse日历比较

In [11]:
import exchange_calendars as xcals

nyse = xcals.get_calendar("XNYS")

expected_dates = nyse.sessions_in_range(
    "2014-01-01",
    "2025-12-31"
).tz_localize(None)

actual_dates = data["SPY"].index.normalize()

missing_dates = expected_dates.difference(actual_dates)
extra_dates = actual_dates.difference(expected_dates)

print("Expected sessions:", len(expected_dates))
print("Actual SPY rows:", len(actual_dates))
print("Missing sessions:", missing_dates.tolist())
print("Extra dates:", extra_dates.tolist())

Expected sessions: 3018
Actual SPY rows: 3018
Missing sessions: []
Extra dates: []


### 拆股与分红检查

In [12]:
actions_summary = []

for ticker, df in data.items():
    actions_summary.append({
        "Ticker": ticker,
        "Dividend Events": (df["Dividends"] != 0).sum(),
        "Total Dividends": df["Dividends"].sum(),
        "Split Events": (df["Stock Splits"] != 0).sum()
    })

actions_df = pd.DataFrame(actions_summary)
display(actions_df)

,Ticker,Dividend Events,Total Dividends,Split Events
0,AAPL,48,9.158929,2
1,MSFT,48,25.310000,0
2,NVDA,48,0.214025,2
3,GOOGL,7,1.430000,2
4,META,8,4.100000,0
5,AMZN,0,0.000000,1
6,JPM,48,38.400000,0
7,BAC,48,7.520000,0
8,GS,48,74.100000,0
9,V,48,14.715000,1


In [13]:
split_records = []

for ticker, df in data.items():
    splits = df.loc[df["Stock Splits"] != 0, "Stock Splits"]

    for date, ratio in splits.items():
        split_records.append({
            "Ticker": ticker,
            "Date": date,
            "Split Ratio": ratio
        })

splits_df = pd.DataFrame(split_records)

display(splits_df)

,Ticker,Date,Split Ratio
0,AAPL,2014-06-09,7.000
1,AAPL,2020-08-31,4.000
2,NVDA,2021-07-20,4.000
3,NVDA,2024-06-10,10.000
4,GOOGL,2014-04-03,1.998
5,GOOGL,2022-07-18,20.000
6,AMZN,2022-06-06,20.000
7,V,2015-03-19,4.000
8,MA,2014-01-22,10.000
9,PFE,2020-11-17,1.054


### 检查拆股当天的价格收益

In [15]:
for ticker, df in data.items():
    split_dates = df.index[df["Stock Splits"] != 0]

    if len(split_dates) == 0:
        continue

    close_returns = df["Close"].pct_change()

    for date in split_dates:
        print(
            ticker,
            date.date(),
            "Split:", df.loc[date, "Stock Splits"],
            "Close Return:", round(close_returns.loc[date], 4)
        )

AAPL 2014-06-09 Split: 7.0 Close Return: 0.016
AAPL 2020-08-31 Split: 4.0 Close Return: 0.0339
NVDA 2021-07-20 Split: 4.0 Close Return: -0.0089
NVDA 2024-06-10 Split: 10.0 Close Return: 0.0075
GOOGL 2014-04-03 Split: 1.998 Close Return: 0.006
GOOGL 2022-07-18 Split: 20.0 Close Return: -0.0246
AMZN 2022-06-06 Split: 20.0 Close Return: 0.0199
V 2015-03-19 Split: 4.0 Close Return: -0.0016
MA 2014-01-22 Split: 10.0 Close Return: 0.0177
PFE 2020-11-17 Split: 1.054 Close Return: 0.0176
WMT 2024-02-26 Split: 3.0 Close Return: 0.0185


### 验证Adj Close的分红复权规则

In [ ]:
# 用aapl 2014-02-06的分红事件来验证调整因子计算的正确性

aapl = data["AAPL"].copy()

aapl["Factor"] = aapl["Adj Close"] / aapl["Close"]

date = pd.Timestamp("2014-02-06")
previous_date = aapl.index[aapl.index.get_loc(date) - 1]

dividend = aapl.loc[date, "Dividends"]
previous_close = aapl.loc[previous_date, "Close"]

theoretical_factor = 1 - dividend / previous_close

observed_factor = (
    aapl.loc[previous_date, "Factor"]
    / aapl.loc[date, "Factor"]
)

print("Dividend:", dividend)
print("Theoretical factor:", theoretical_factor)
print("Observed factor:", observed_factor)
print("Difference:", observed_factor - theoretical_factor)

Dividend: 0.108929
Theoretical factor: 0.9940498019434826
Observed factor: 0.9940498992864761
Difference: 9.734299355468323e-08


In [18]:
#扩展到全部分红事件

validation = []

for ticker, df in data.items():
    factor = df["Adj Close"] / df["Close"]

    theoretical = 1 - (
        df["Dividends"] / df["Close"].shift(1)
    )

    observed = factor.shift(1) / factor

    mask = (
        (df["Dividends"] > 0)
        & (df["Stock Splits"] == 0)
    )

    for date in df.index[mask]:
        validation.append({
            "Ticker": ticker,
            "Date": date,
            "Theoretical": theoretical.loc[date],
            "Observed": observed.loc[date],
            "Difference": (
                observed.loc[date] - theoretical.loc[date]
            )
        })

validation_df = pd.DataFrame(validation)

display(
    validation_df.assign(
        Abs_Difference=validation_df["Difference"].abs()
    ).sort_values("Abs_Difference", ascending=False).head(20)
)

,Ticker,Date,Theoretical,Observed,Difference,Abs_Difference
256,GS,2014-05-28,0.996600,0.996599,-7.104988e-07,7.104988e-07
800,XOM,2017-05-10,0.990645,0.990644,-6.939144e-07,6.939144e-07
366,MA,2017-10-05,0.998452,0.998452,6.516462e-07,6.516462e-07
175,JPM,2018-01-04,0.994818,0.994818,6.203962e-07,6.203962e-07
680,COST,2023-08-24,0.998114,0.998115,6.048545e-07,6.048545e-07
460,UNH,2017-06-15,0.995887,0.995887,-5.989585e-07,5.989585e-07
648,COST,2016-02-10,0.997296,0.997297,5.904701e-07,5.904701e-07
407,JNJ,2016-02-19,0.992805,0.992805,-5.505441e-07,5.505441e-07
308,V,2015-05-13,0.998253,0.998252,-5.468193e-07,5.468193e-07
703,PG,2017-01-18,0.992137,0.992137,-5.453509e-07,5.453509e-07


### 极端日收益和OHLC完全相同检验

In [19]:
summary = []
extreme_records = []
same_ohlc_records = []

for ticker, df in data.items():
    df = df.sort_index().copy()

    # 1. Adj Close 日收益率
    daily_return = df["Adj Close"].pct_change(fill_method=None)

    # 极端收益阈值：20%
    extreme_mask = daily_return.abs() > 0.20

    # 2. OHLC 四个价格完全相同
    same_ohlc_mask = (
        (df["Open"] == df["High"])
        & (df["High"] == df["Low"])
        & (df["Low"] == df["Close"])
    )

    # 3. 汇总
    summary.append({
        "Ticker": ticker,
        "Extreme Return Days": int(extreme_mask.sum()),
        "Same OHLC Days": int(same_ohlc_mask.sum())
    })

    # 4. 保存极端收益记录
    if extreme_mask.any():
        records = df.loc[extreme_mask].copy()
        records["Ticker"] = ticker
        records["Daily Return"] = daily_return.loc[extreme_mask]
        extreme_records.append(records)

    # 5. 保存 OHLC 相同记录
    if same_ohlc_mask.any():
        records = df.loc[same_ohlc_mask].copy()
        records["Ticker"] = ticker
        same_ohlc_records.append(records)

# 汇总表
quality_summary = pd.DataFrame(summary)

# 极端收益明细
extreme_returns_df = (
    pd.concat(extreme_records)
    if extreme_records else pd.DataFrame()
)

# OHLC 相同明细
same_ohlc_df = (
    pd.concat(same_ohlc_records)
    if same_ohlc_records else pd.DataFrame()
)

display(quality_summary)

,Ticker,Extreme Return Days,Same OHLC Days
0,AAPL,0,0
1,MSFT,0,0
2,NVDA,2,0
3,GOOGL,0,0
4,META,4,0
5,AMZN,0,0
6,JPM,0,0
7,BAC,0,0
8,GS,0,0
9,V,0,0


In [20]:
if not extreme_returns_df.empty:
    extreme_returns_df = extreme_returns_df.assign(
        Abs_Return=extreme_returns_df["Daily Return"].abs()
    ).sort_values("Abs_Return", ascending=False)

    display(
        extreme_returns_df[
            ["Ticker", "Adj Close", "Daily Return", "Abs_Return"]
        ]
    )

,Ticker,Adj Close,Daily Return,Abs_Return
Date,,,,
2016-11-11,NVDA,2.158154,0.298067,0.298067
2022-02-03,META,235.504425,-0.263901,0.263901
2022-10-27,META,97.010857,-0.245571,0.245571
2023-05-25,NVDA,37.859867,0.243696,0.243696
2023-02-02,META,186.979172,0.232824,0.232824
2025-04-17,UNH,436.460297,-0.223797,0.223797
2024-02-02,META,470.483856,0.203176,0.203176
